# New Tokyo PoC Script

## 1 · Scene & Environment Setup

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # Use GPU 0, will use CPU otherwise
import numpy as np
import pandas as pd
import json
import csv as _csv
import poc.setup as poc
import poc.utils as utils
import poc.utils_numpy as nputils
import core.rt as rt

path_suffix = "/home/polimi"

# Antennas are Car 1: [1, 2], RSU: [30, 31, 40], Car 2: [5, 6, 7]
transmitters = [30, 31, 5, 6]
receivers = [1, 2, 40, 7]

# ── Ray-tracing scene ─────────────────────────────────────────────────────────
scene = poc.setup_scene(
    file_name=f"{path_suffix}/tokyo-polimi-dt-jsac/scenarios/ookayama_full_flat/ookayama.xml",
    frequency=60e9,
    bandwidth=1000e6,
    verbose=False,
    time_checker=True
)

poc.setup_antenna_type(
    transmitters=transmitters,
    receivers=receivers,
    pattern="panasonic_wigig_rsu",
    elevation_csv=f"{path_suffix}/tokyo-polimi-dt-jsac/panasonic-rsu-pattern/elevation_beam_16.csv",
    azimuth_csv=f"{path_suffix}/tokyo-polimi-dt-jsac/panasonic-rsu-pattern/azimuth_beam_16.csv",
    polarization="H", # Use horizontal (define how is mounted in setup_antenna_on_object)
    simulate_perfect_beamforming=True
)

poc.setup_rt(
    time_checker=True,
    rt_diffraction=False,
    rt_corner_diffraction=False,
    rt_max_depth=4,
)

poc.setup_filters(
    transmitters=transmitters,
    receivers=receivers,
    use_kalman_filter=True,
    kalman_process_var=0.3,
    kalman_meas_var=0.8, # Lower = trust measurement more
    kalman_rt_var=3      # Higher = trust RT less
)

struc = poc.startup()

    [INFO] Time checker is enabled.
    [INFO] Loaded scene with frequency 60.0 GHz, bandwidth 1000.0 MHz.
    [INFO] Expecting UDP messages from Tokyo Digital Twin on UDP/35944
    [INFO] Setup procedure complete.


## 2 · Object configurator

In [2]:
# ── Asset paths & TX powers ───────────────────────────────────────────────────
CAR_1_MESH       = f"{path_suffix}/tokyo-polimi-dt-jsac/obj_meshes/coms.ply"
CAR_2_MESH       = f"{path_suffix}/tokyo-polimi-dt-jsac/obj_meshes/estima.ply"
RSU_MESH         = f"{path_suffix}/tokyo-polimi-dt-jsac/obj_meshes/rsu.ply"
TREE_MESH        = f"{path_suffix}/tokyo-polimi-dt-jsac/obj_meshes/tree.ply"


# Add objects to the scene
poc.add_object(ref_obj_id=1, mesh_path=CAR_1_MESH, position=[193.263, 182.911, 0.758495])
poc.add_object(ref_obj_id=2, mesh_path=CAR_2_MESH, position=[213.124, 191.882, 0.874555])
poc.add_object(ref_obj_id=101, mesh_path=RSU_MESH, position=[195.940, 197.486, 5.0/2])

trees = {}
trees[1] = [151.070, 137.283, 8.64/2]
trees[2] = [167.037, 135.432, 8.64/2]
trees[3] = [153.032, 149.437, 8.64/2]
trees[4] = [168.768, 147.313, 8.64/2]
trees[5] = [170.476, 159.486, 8.64/2]
trees[6] = [185.203, 157.361, 8.64/2]
trees[7] = [186.522, 160.437, 8.64/2]

for i in range(1, len(trees) + 1):
    poc.add_tree(ref_tree_id=i, mesh_path=TREE_MESH, position=trees[i])

# Add antennas to each object
# Peers (Tx)-(Rx) are 30->1, 30->7, 6->40, 5->2
poc.setup_antenna_on_object(ref_obj_id=1, 
                            ant_id=1, peer_antenna_id=30, # V2I Client (Rx)
                            displacement=[0.096, 0.157001, 0.791505], orientation=[0, 0, 0], mounted_vertically=True)

poc.setup_antenna_on_object(ref_obj_id=1,  
                            ant_id=2, peer_antenna_id=5,  # V2V Client (Rx)
                            displacement=[-0.281679, 0.477371, 0.975445], orientation=[0, 0, 0], mounted_vertically=False)

poc.setup_antenna_on_object(ref_obj_id=101,
                            ant_id=30, peer_antenna_id=1, # I2V Server (Tx)
                            displacement=[0, 0, 5.0/2 + 0.5], orientation=np.array([np.radians(220), np.radians(10), 0]), mounted_vertically=False,
                            tx_power_dbm=16)

poc.setup_antenna_on_object(ref_obj_id=101, 
                            ant_id=31, peer_antenna_id=7, # I2V Server (Tx)
                            displacement=[0, 0, 5.0/2 + 0.5], orientation=np.array([np.radians(220), np.radians(10), 0]), mounted_vertically=False,
                            tx_power_dbm=16)

poc.setup_antenna_on_object(ref_obj_id=101, 
                            ant_id=40, peer_antenna_id=6, # I2V Client (Rx)
                            displacement=[0, 0, 5.0/2 + 0.5], orientation=np.array([np.radians(-45), np.radians(10), 0]), mounted_vertically=False)

poc.setup_antenna_on_object(ref_obj_id=2, 
                            ant_id=5, peer_antenna_id=2,  # V2V Server (Tx)
                            displacement=[0.096, 0.157001, 0.791505], orientation=[0, 0, 0], mounted_vertically=True,
                            tx_power_dbm=16)

poc.setup_antenna_on_object(ref_obj_id=2, 
                            ant_id=6, peer_antenna_id=40, # V2I Server (Tx)
                            displacement=[-0.281679, 0.477371, 0.975445], orientation=[0, 0, 0], mounted_vertically=False,
                            tx_power_dbm=16)

poc.setup_antenna_on_object(ref_obj_id=2, 
                            ant_id=7, peer_antenna_id=31, # V2I Client (Rx)
                            displacement=[0.1, 0.1, 0.8], orientation=[0, 0, 0], mounted_vertically=True)

In [3]:
utils.move_object(ref_obj_id=1, 
                  position=[192.584, 189.436, 0.758495], 
                  heading_angle=0, 
                  velocity=5,
                  sionna_structure=struc)

utils.move_object(ref_obj_id=2, 
                  position=[201.080, 188.896, 0.975445], 
                  heading_angle=190, 
                  velocity=2,
                  sionna_structure=struc)

     [TIME] Time taken for beamforming check and orientation update: 20.9014 ms
     [TIME] Time taken for beamforming check and orientation update: 0.8547 ms
    [TIME] Time taken for location updates: 46.7336 ms
     [TIME] Time taken for beamforming check and orientation update: 0.6438 ms
     [TIME] Time taken for beamforming check and orientation update: 16.1038 ms
     [TIME] Time taken for beamforming check and orientation update: 55.7000 ms
    [TIME] Time taken for location updates: 78.1995 ms


In [ ]:
import poc.light_rt as rt

In [ ]:
#scene.get("ant_3").orientation = np.array([np.radians(220), np.radians(10), 0]) # TX RED
#scene.get("ant_4").orientation = np.array([np.radians(-45), np.radians(10), 0]) # RX GREEN

scene.preview(show_orientations=True, resolution=(800, 600))

In [19]:
def compute_rays(sionna_structure):

    # Compute paths
    paths = sionna_structure["path_solver"](scene=sionna_structure["scene"],
                                            max_depth=sionna_structure["max_depth"],
                                            los=sionna_structure["los"],
                                            specular_reflection=sionna_structure["specular_reflection"],
                                            diffuse_reflection=sionna_structure["diffuse_reflection"],
                                            refraction=sionna_structure["refraction"],
                                            synthetic_array=sionna_structure["synthetic_array"],
                                            seed=sionna_structure["seed"],
                                            diffraction=sionna_structure["diffraction"],
                                            edge_diffraction=sionna_structure["corner_diffraction"])
    paths.normalize_delays = False

    # Save raw paths (for debugging and analysis)
    sionna_structure["paths"] = paths

    return paths


paths = compute_rays(struc)

a_real, a_imag = paths.a
path_coefficients = a_real.numpy() + 1j * a_imag.numpy()

transmitters = [30, 31, 5, 6]
receivers = [1, 2, 40, 7]

tx_to_idx = {tx_id: i for i, tx_id in enumerate(transmitters)}
rx_to_idx = {rx_id: i for i, rx_id in enumerate(receivers)}

# path_coefficients shape: [num_rx, num_rx_ant, num_tx, num_tx_ant, num_paths]
links = [(30, 1), (30, 7), (6, 40), (5, 2)]

link_paths = {}
for tx_id, rx_id in links:
    ti = tx_to_idx[tx_id]
    ri = rx_to_idx[rx_id]
    coeffs = path_coefficients[ri, 0, ti, 0, :]
    # Filter out zero-padded paths
    active = coeffs[coeffs != 0]
    link_paths[(tx_id, rx_id)] = coeffs
    print(f"Tx {tx_id} -> Rx {rx_id}: {len(active)} active paths out of {len(coeffs)}")
    print(f"  Coefficients: {active}\n")



Tx 30 -> Rx 1: 4 active paths out of 64
  Coefficients: [-1.1391024e-05+0.0000000e+00j -1.3343153e-07+1.1503176e-08j
  2.3500439e-08-2.0732096e-09j -1.9942316e-08+1.2817165e-09j]

Tx 30 -> Rx 7: 0 active paths out of 64
  Coefficients: []

Tx 6 -> Rx 40: 33 active paths out of 64
  Coefficients: [-1.0022026e-05+3.5035481e-07j  4.3563600e-06-8.8553819e-08j
  4.3559480e-06-8.8534676e-08j  9.3060798e-06-3.5704889e-07j
 -3.9867014e-06+9.4819669e-08j -3.9870260e-06+9.4835528e-08j
  8.2266279e-06-2.8791865e-07j  8.2248498e-06-2.8785672e-07j
  8.2186925e-06-2.8764327e-07j  8.1757362e-06-2.8625414e-07j
  8.1812641e-06-2.8644664e-07j  8.1791304e-06-2.8637174e-07j
  7.8086332e-06-2.7329915e-07j  8.1782118e-06-2.8633886e-07j
  8.1718863e-06-2.8611313e-07j  8.1785647e-06-2.8635117e-07j
  7.6826482e-06-2.6894543e-07j  8.1652324e-06-2.8588607e-07j
 -4.0064047e-06+7.8864488e-08j -4.0061773e-06+7.8869853e-08j
 -4.0064174e-06+7.8865888e-08j -4.0122131e-06+7.8932615e-08j
 -4.0065056e-06+7.8876518e-08j -

## 3 · Showtime!

In [ ]:
import time
import json
import random
import numpy as np
import csv

udp_socket = struc["udp_socket"]
verbose = struc["verbose"]

def main():
    while True:
        # Receive data from the socket
        payload, address = udp_socket.recvfrom(1024*10)
        struc["latest_msg_address"] = address
        message = payload.decode()
        
        t_tot = time.time()
        # Parse the JSON message
        data = json.loads(message)

        if isinstance(data, dict) and "data" in data and isinstance(data["data"], list):
            msg_entries = data["data"]
        elif isinstance(data, list):
            msg_entries = data
        else:
            msg_entries = [data]

        # Extract message type (if available)
        type_val = None
        if len(msg_entries) > 0 and isinstance(msg_entries[0], dict):
            type_val = msg_entries[0].get("type")
        
        # Is a configuration message
        if isinstance(type_val, str) and "RT_CONFIGURATION" in type_val.upper():
            rt.manage_online_reconfiguration(msg_entries, struc, is_manual_override=False)
            continue
        
        # Is data --> Handle prediction request
        if isinstance(data, dict) and "data" in data and isinstance(data["data"], list):
            data = data["data"]

        if verbose:
            print(" ")
            print("  - - - - - - - - - - - - - - - - -  NEW PREDICTION REQUEST  - - - - - - - - - - - - - - - - -  ")
            print(" ")

        # Extract current measured RSSI to calibrate the Digital Twin                        
        current_1vs2 = data[0]["data"]["RSSI_Vehicle1_2"] if data[0]["data"]["RSSI_Vehicle1_2"] != 0 else None
        current_1vs3 = data[0]["data"]["RSSI_Vehicle1_3"] if data[0]["data"]["RSSI_Vehicle1_3"] != 0 else None
        current_2vs3 = data[0]["data"]["RSSI_Vehicle2_3"] if data[0]["data"]["RSSI_Vehicle2_3"] != 0 else None
        
        if struc["use_kalman_filter"]:
            for key, filt in struc["filters"].items():
                filt.predict()
        

        # Update the locations in the scenario
        future_timestamp = data[1]["clock"]
        predicted_p1 = data[1]["data"]["x1"]
        predicted_p2 = data[1]["data"]["x2"]
        predicted_p3 = data[1]["data"]["x3"]
        t = time.time()

        rt.manage_location_message(f"LOC_UPDATE:veh2,{predicted_p2['x']},{predicted_p2['z']},{0.0},{predicted_p2['yaw']},0,0,0", struc)
        rt.manage_location_message(f"LOC_UPDATE:veh3,{predicted_p1['x']},{predicted_p1['z']},{0.0},{predicted_p1['yaw']},0,0,0", struc) # I know they are inverted!
        rt.manage_location_message(f"LOC_UPDATE:veh1,{predicted_p3['x']},{predicted_p3['z']},{0.0},{predicted_p3['yaw']},0,0,0", struc) # I know they are inverted!
    
        location_update_time = (time.time() - t) * 1000
        if struc["time_checker"]:
            print(f"        Location update took: {location_update_time:.2f} ms")

        # Compute filtered RSSI predictions
        t_c = time.time()
        raw_rssi_1vs2 = {}
        raw_rssi_1vs3 = {}
        raw_rssi_2vs3 = {}
        jit = struc["montecarlo_max_position_jitter"]
        
        is_monte_carlo_enabled = struc["montecarlo_realizations"] > 1 and struc["montecarlo_max_position_jitter"] > 0.0
        real = struc["montecarlo_realizations"] if is_monte_carlo_enabled else 1

        for i in range(real):
            if is_monte_carlo_enabled:
                struc["seed"] = random.randint(0, int(1e6))
            print(f"        RSSI Prediction iteration {i+1}/{real} with seed {struc['seed']}")
            manage_location_message(f"LOC_UPDATE:veh2,{predicted_p2['x'] + random.uniform(-jit, jit)},{predicted_p2['z'] + random.uniform(-jit, jit)},{0.0},{predicted_p2['yaw']},0,0,0", struc)
            raw_rssi_1vs2[i] = tx_power - get_path_loss("car_1", "car_2", struc) + correction - 2.5
            los_1_2 = get_los("car_1", "car_2", struc)
            raw_rssi_1vs3[i] = tx_power - get_path_loss("car_1", "car_3", struc) + correction + 7 + 1.80 - 0.25
            los_1_3 = get_los("car_1", "car_3", struc)
            raw_rssi_2vs3[i] = tx_power - get_path_loss("car_2", "car_3", struc) + correction
            los_2_3 = get_los("car_2", "car_3", struc)
            struc["rays_cache"] = {}  # Clear rays cache to force re-computation
            i += 1

        raw_rssi_1vs2 = sum(raw_rssi_1vs2.values()) / real
        raw_rssi_1vs3 = sum(raw_rssi_1vs3.values()) / real
        raw_rssi_2vs3 = sum(raw_rssi_2vs3.values()) / real

        if struc["use_kalman_filter"]:
            # Update Kalman filters
            rssi_1vs2 = struc["filters"][("1","2")].update(z_meas=current_1vs2, z_rt=raw_rssi_1vs2)
            rssi_1vs3 = struc["filters"][("1","3")].update(z_meas=current_1vs3, z_rt=raw_rssi_1vs3)
            rssi_2vs3 = struc["filters"][("2","3")].update(z_meas=current_2vs3, z_rt=raw_rssi_2vs3)
        
        if struc["use_adaptive_bias_filter"]:
            rssi_1vs2 = struc["filters"][("1","2")].step(predicted_rt=raw_rssi_1vs2, current_meas=current_1vs2)
            rssi_1vs3 = struc["filters"][("1","3")].step(predicted_rt=raw_rssi_1vs3, current_meas=current_1vs3)
            rssi_2vs3 = struc["filters"][("2","3")].step(predicted_rt=raw_rssi_2vs3, current_meas=current_2vs3)

        rssi_prediction_time = (time.time() - t_c) * 1000
        if struc["time_checker"]:
            print(f"        RT and RSSI extraction took: {rssi_prediction_time:.2f} ms")

        if struc["verbose"]:
            print(f"        Raw predicted RSSI:   1vs2={raw_rssi_1vs2:.2f} dBm, 1vs3={raw_rssi_1vs3:.2f} dBm, 2vs3={raw_rssi_2vs3:.2f} dBm")
            print(f"        Filtered RSSI:        1vs2={rssi_1vs2:.2f} dBm, 1vs3={rssi_1vs3:.2f} dBm, 2vs3={rssi_2vs3:.2f} dBm")

        # Prepare the response
        response = [{}]
        response[0]["clock"] = future_timestamp
        response[0]["data"] = {}
        response[0]["data"]["x1"] = predicted_p1
        response[0]["data"]["x2"] = predicted_p2
        response[0]["data"]["x3"] = predicted_p3
        response[0]["data"]["RSSI_Vehicle1_2"] = rssi_1vs2
        response[0]["data"]["RSSI_Vehicle1_3"] = rssi_1vs3
        response[0]["data"]["RSSI_Vehicle2_3"] = rssi_2vs3
        response[0]["data"]["raw_RSSI_Vehicle1_2"] = raw_rssi_1vs2
        response[0]["data"]["raw_RSSI_Vehicle1_3"] = raw_rssi_1vs3
        response[0]["data"]["raw_RSSI_Vehicle2_3"] = raw_rssi_2vs3

        response = json.dumps(response, default=lambda o: float(o) if isinstance(o, np.float32) else o)

        # Send back the response
        #udp_socket.sendto(response.encode(), ("20.0.0.10", 35944))
        udp_socket.sendto(response.encode(), address) # Use this for local testing script
        total_elapsed_time = (time.time() - t_tot) * 1000

        if verbose:
            print(" ")
            print(f"  - - - - - - - - - - - - - - - - -   Handled in {total_elapsed_time:.2f} ms   - - - - - - - - - - - - - - - - -  ")
            print(" ")
            print(" ")
        
        # Logging
        local_timestamp = time.time()
        with open(struc["log_file"], mode="a", newline="") as file:
            writer = csv.writer(file)
            writer.writerow([local_timestamp, data[0]["clock"], future_timestamp,
                             message,
                             struc["sionna_location_db"][1]['x'], struc["sionna_location_db"][1]['y'], struc["sionna_location_db"][1]['angle'],
                             struc["sionna_location_db"][2]['x'], struc["sionna_location_db"][2]['y'], struc["sionna_location_db"][2]['angle'],
                             struc["sionna_location_db"][3]['x'], struc["sionna_location_db"][3]['y'], struc["sionna_location_db"][3]['angle'],
                             raw_rssi_1vs2, raw_rssi_1vs3, raw_rssi_2vs3, 
                             rssi_1vs2, rssi_1vs3, rssi_2vs3,
                             location_update_time, rssi_prediction_time, total_elapsed_time, los_1_2, los_1_3, los_2_3])
        
        '''
        from sionna.rt import render_to_file
        # Frame-by-frame exporter
        global frame_num
        print("Exporting frame ", frame_num)
        
        struc["scene"].render_to_file(camera=my_cam, 
                                         filename=f"/home/rpegurri/Tokyo NDT Integration/PoC Tokyo Science/export/frame_{frame_num}.png", 
                                         show_devices=False)
        frame_num += 1
        '''
        

        if message.startswith("SHUTDOWN_SIONNA"):
            print("Got SHUTDOWN_SIONNA message. Bye!")
            udp_socket.close()
            break

# Entry point
if __name__ == "__main__":
    main()

# New showtime

In [ ]:
import time
import json
import numpy as np
import csv

udp_socket = struc["udp_socket"]
verbose = struc["verbose"]

# ── Configuration for correction factors per TX-RX pair ────────────────────────
# Adjust these based on calibration
correction_factors = {
    (1, 2): -2.5,
    (1, 3): 7 + 1.80 - 0.25,
    (2, 3): 0.0,
}

# Extract TX powers for each transmitter
tx_powers = {tx_id: struc["tx_powers"].get(tx_id, 10) for tx_id in transmitters}

# Generate CSV header dynamically
csv_header = ["local_timestamp", "data_clock", "future_timestamp", "message"]
for tx_id in transmitters:
    csv_header.extend([f"pos_{tx_id}_x", f"pos_{tx_id}_y", f"pos_{tx_id}_angle"])
for tx_id in transmitters:
    for rx_id in receivers:
        csv_header.append(f"raw_RSSI_{tx_id}_{rx_id}")
for tx_id in transmitters:
    for rx_id in receivers:
        csv_header.append(f"RSSI_{tx_id}_{rx_id}")
csv_header.extend(["location_update_time_ms", "rssi_prediction_time_ms", "total_time_ms"])
for tx_id in transmitters:
    for rx_id in receivers:
        csv_header.append(f"LoS_{tx_id}_{rx_id}")

# Write header if log file is empty
if os.path.getsize(struc["log_file"]) == 0:
    with open(struc["log_file"], mode="a", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(csv_header)

def main():
    while True:
        # Receive data from the socket
        payload, address = udp_socket.recvfrom(1024*10)
        struc["latest_msg_address"] = address
        message = payload.decode()
        
        t_tot = time.time()
        # Parse the JSON message
        data = json.loads(message)

        if isinstance(data, dict) and "data" in data and isinstance(data["data"], list):
            msg_entries = data["data"]
        elif isinstance(data, list):
            msg_entries = data
        else:
            msg_entries = [data]

        # Extract message type (if available)
        type_val = None
        if len(msg_entries) > 0 and isinstance(msg_entries[0], dict):
            type_val = msg_entries[0].get("type")
        
        # Is a configuration message
        if isinstance(type_val, str) and "RT_CONFIGURATION" in type_val.upper():
            rt.manage_online_reconfiguration(msg_entries, struc, is_manual_override=False)
            continue
        
        # Is data --> Handle prediction request
        if isinstance(data, dict) and "data" in data and isinstance(data["data"], list):
            data = data["data"]

        if verbose:
            print(" ")
            print("  - - - - - - - - - - - - - - - - -  NEW PREDICTION REQUEST  - - - - - - - - - - - - - - - - -  ")
            print(" ")

        # Extract future timestamp and predicted positions for all transmitters
        future_timestamp = data[1]["clock"]
        predicted_positions = {}
        
        for tx_id in transmitters:
            # Try to extract position data (handle both dict and array indexing)
            if f"x{tx_id}" in data[1]["data"]:
                predicted_positions[tx_id] = data[1]["data"][f"x{tx_id}"]
            elif str(tx_id) in data[1]["data"]:
                predicted_positions[tx_id] = data[1]["data"][str(tx_id)]
            else:
                raise KeyError(f"Position data for transmitter {tx_id} not found in message")
        
        # Extract current measured RSSI for all TX-RX pairs
        current_meas = {}
        for tx_id in transmitters:
            for rx_id in receivers:
                key = f"RSSI_{tx_id}_{rx_id}"
                if key in data[0]["data"]:
                    val = data[0]["data"][key]
                    current_meas[(tx_id, rx_id)] = val if val != 0 else None
                else:
                    current_meas[(tx_id, rx_id)] = None
        
        # Kalman filter predict step
        if struc["use_kalman_filter"]:
            for key, filt in struc["filters"].items():
                filt.predict()
        
        # Update locations in the scenario for all transmitters
        t = time.time()
        for tx_id, pos in predicted_positions.items():
            rt.manage_location_message(
                f"LOC_UPDATE:{tx_id},{pos['x']},{pos['z']},{0.0},{pos['yaw']},0,0,0", 
                struc
            )
        location_update_time = (time.time() - t) * 1000
        if struc["time_checker"]:
            print(f"        Location update took: {location_update_time:.2f} ms")

        # Compute RSSI predictions for all TX-RX pairs
        t_c = time.time()
        raw_rssi = {}
        filtered_rssi = {}
        los = {}
        
        for tx_id in transmitters:
            for rx_id in receivers:
                # Look up TX power and correction factor
                tx_power = tx_powers[tx_id]
                correction = correction_factors.get((tx_id, rx_id), 0.0)
                
                # Compute raw RSSI
                path_loss = rt.get_path_loss(tx_id, rx_id, struc)
                raw_rssi[(tx_id, rx_id)] = tx_power - path_loss + correction
                
                # Compute LoS
                los[(tx_id, rx_id)] = rt.get_los(tx_id, rx_id, struc)
        
        rssi_prediction_time = (time.time() - t_c) * 1000
        if struc["time_checker"]:
            print(f"        RT and RSSI extraction took: {rssi_prediction_time:.2f} ms")

        # Apply filters to RSSI predictions
        for (tx_id, rx_id), raw_val in raw_rssi.items():
            meas_val = current_meas[(tx_id, rx_id)]
            
            if struc["use_kalman_filter"]:
                filtered_rssi[(tx_id, rx_id)] = struc["filters"][(str(tx_id), str(rx_id))].update(z_meas=meas_val, z_rt=raw_val)
            elif struc["use_adaptive_bias_filter"]:
                filtered_rssi[(tx_id, rx_id)] = struc["filters"][(str(tx_id), str(rx_id))].step(predicted_rt=raw_val, current_meas=meas_val)
            else:
                filtered_rssi[(tx_id, rx_id)] = raw_val

        if verbose:
            print("        Raw predicted RSSI:")
            for (tx_id, rx_id), val in raw_rssi.items():
                print(f"          {tx_id}→{rx_id}: {val:.2f} dBm")
            print("        Filtered RSSI:")
            for (tx_id, rx_id), val in filtered_rssi.items():
                print(f"          {tx_id}→{rx_id}: {val:.2f} dBm")

        # Prepare the response dynamically
        response = [{"clock": future_timestamp, "data": {}}]
        
        # Add positions
        for tx_id, pos in predicted_positions.items():
            response[0]["data"][f"x{tx_id}"] = pos
        
        # Add raw RSSI
        for (tx_id, rx_id), val in raw_rssi.items():
            response[0]["data"][f"raw_RSSI_{tx_id}_{rx_id}"] = val
        
        # Add filtered RSSI
        for (tx_id, rx_id), val in filtered_rssi.items():
            response[0]["data"][f"RSSI_{tx_id}_{rx_id}"] = val

        response_json = json.dumps(response, default=lambda o: float(o) if isinstance(o, np.float32) else o)

        # Send back the response
        udp_socket.sendto(response_json.encode(), address)
        total_elapsed_time = (time.time() - t_tot) * 1000

        if verbose:
            print(" ")
            print(f"  - - - - - - - - - - - - - - - - -   Handled in {total_elapsed_time:.2f} ms   - - - - - - - - - - - - - - - - -  ")
            print(" ")

        # Logging
        local_timestamp = time.time()
        with open(struc["log_file"], mode="a", newline="") as file:
            writer = csv.writer(file)
            
            # Build row dynamically
            row = [local_timestamp, data[0]["clock"], future_timestamp, message]
            
            # Add position data
            for tx_id in transmitters:
                row.extend([
                    struc["sionna_location_db"][tx_id]['x'],
                    struc["sionna_location_db"][tx_id]['y'],
                    struc["sionna_location_db"][tx_id]['angle']
                ])
            
            # Add raw RSSI
            for tx_id in transmitters:
                for rx_id in receivers:
                    row.append(raw_rssi[(tx_id, rx_id)])
            
            # Add filtered RSSI
            for tx_id in transmitters:
                for rx_id in receivers:
                    row.append(filtered_rssi[(tx_id, rx_id)])
            
            # Add timing
            row.extend([location_update_time, rssi_prediction_time, total_elapsed_time])
            
            # Add LoS
            for tx_id in transmitters:
                for rx_id in receivers:
                    row.append(los[(tx_id, rx_id)])
            
            writer.writerow(row)

        if message.startswith("SHUTDOWN_SIONNA"):
            print("Got SHUTDOWN_SIONNA message. Bye!")
            udp_socket.close()
            break

# Entry point
if __name__ == "__main__":
    main()